# 🪱 JORINOVA NEXUS — Stool/Urine Helminths (worm eggs / O&P) detector (fine-tuning)

Fine-tunes a **YOLOv8** detector to find helminth **ova / larvae** (roundworms, tapeworms,
flukes) on stool or urine microscopy and produces `helminths.pt` for the app's vision
service. Each egg is mapped to its **disease** by the app. *(Intestinal protozoa —
Entamoeba, Giardia… — have their own `protozoa` detector.)*

> **Fine-tuning, NOT from scratch** — starts from pretrained weights
> (`yolov8s.pt`, or your blood-domain `malaria.pt`) and adapts them.

**Before you start:** Runtime → **Change runtime type** → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)
No GitHub token needed. Weights + runs are saved to `MyDrive/nexus_helminths/` so a disconnect won't lose them.

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_helminths', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_helminths')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data — Option A (Roboflow, ready) or Option B (Kaggle Chula)
Option A gives a ready YOLOv8 export (recommended). Option B is the full 11k Chula set.

### ✅ Option A — Roboflow Chula-11 parasite eggs — **ready, no blanks, not Kaggle**
Concrete project **`parasites-onxa6/parasites-trgfz`** (11 helminth-egg species WITH bounding boxes, ~10k imgs). Uses your free Roboflow key (roboflow.com → Settings → API). Auto-picks a version and **renames classes to the app keys** in `helminths_organisms.json`. Run it, then go to step 5.

*(Skip Option B / Kaggle below — the Roboflow cell replaces it.)*

In [ ]:
# ── Data: Roboflow Chula-11 parasite eggs (keyless of Kaggle) -> DATA_YAML ──
import os, yaml
from getpass import getpass
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

proj = rf.workspace('parasites-onxa6').project('parasites-trgfz')   # 11 species, bounding boxes
ds = None
for v in range(1, 12):
    try:
        ds = proj.version(v).download('yolov8', location='/content/data/para_rf'); print('OK v', v); break
    except Exception:
        continue
assert ds, 'download failed — search https://universe.roboflow.com/search?q=parasite%20egg for another project.'
DATA_YAML = ds.location + '/data.yaml'

# rename dataset class names -> app keys (helminths_organisms.json), robust to order/spelling
def canon(n):
    n = str(n).lower()
    for key, out in [('ascaris','ascaris_lumbricoides'), ('capillaria','capillaria_philippinensis'),
                     ('enterobius','enterobius_vermicularis'), ('fasciolopsis','fasciolopsis_buski'),
                     ('hookworm','hookworm'), ('diminuta','hymenolepis_diminuta'), ('nana','hymenolepis_nana'),
                     ('opisthorchis','opisthorchis_viverrini'), ('paragonimus','paragonimus'),
                     ('taenia','taenia'), ('trichuris','trichuris_trichiura')]:
        if key in n: return out
    return n.strip().replace(' ', '_').replace('-', '_')

d = yaml.safe_load(open(DATA_YAML)); names = d['names']
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
d['names'] = [canon(x) for x in names]; d['nc'] = len(d['names'])
yaml.safe_dump(d, open(DATA_YAML, 'w'), sort_keys=False)
print('DATA_YAML =', DATA_YAML, '| classes:', d['names'])

### Option B — Kaggle Chula-ParasiteEgg-11 (11 species, 11k imgs) → COCO→YOLO
Needs `kaggle.json` (kaggle.com → Account → Create New API Token). Uploads it, downloads
the set, then converts COCO boxes to YOLO. Adjust `base` if the unzip layout differs.

In [ ]:
# 1) upload kaggle.json (from your Kaggle account), then download + unzip
# from google.colab import files; files.upload()   # pick kaggle.json
# import os; os.makedirs('/root/.kaggle', exist_ok=True)
# os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !kaggle datasets download -d macharning/chula-parasite-dataset -p /content/data/chula --unzip
#
# 2) COCO -> YOLO conversion (best-effort; inspect the layout if it errors)
import os, glob, json, shutil
from pathlib import Path
from collections import defaultdict
base = '/content/data/chula'
coco_path = None
for j in glob.glob(base + '/**/*.json', recursive=True):
    try:
        d = json.load(open(j))
        if isinstance(d, dict) and {'annotations','images','categories'} <= set(d):
            coco_path = j; break
    except Exception:
        pass
assert coco_path, 'No COCO json found under ' + base + ' — inspect the download layout.'
coco = json.load(open(coco_path))
cat_sorted = sorted(c['id'] for c in coco['categories'])
catname = {c['id']: c['name'] for c in coco['categories']}
names = [str(catname[k]).strip().lower().replace(' ', '_') for k in cat_sorted]
catidx = {k: i for i, k in enumerate(cat_sorted)}
disk = {os.path.basename(p): p for ext in ('*.jpg','*.jpeg','*.png')
        for p in glob.glob(base + '/**/' + ext, recursive=True)}
by_img = defaultdict(list)
for a in coco['annotations']:
    by_img[a['image_id']].append(a)
items = list(coco['images']); OUT = '/content/data/para_yolo'
nval = max(1, len(items)//7)
for k, im in enumerate(items):
    split = 'val' if k < nval else 'train'
    src = disk.get(os.path.basename(im['file_name']))
    if not src: continue
    W, H = im['width'], im['height']
    oi = Path(OUT)/'images'/split; ol = Path(OUT)/'labels'/split
    oi.mkdir(parents=True, exist_ok=True); ol.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, oi/os.path.basename(src))
    lines = []
    for a in by_img.get(im['id'], []):
        x, y, w, h = a['bbox']; cid = catidx[a['category_id']]
        lines.append(f'{cid} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}')
    (ol/(Path(src).stem + '.txt')).write_text(chr(10).join(lines))
DATA_YAML = os.path.join(OUT, 'data.yaml')
open(DATA_YAML, 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\nnc: {len(names)}\nnames: {names}\n')
print('converted ->', DATA_YAML, '| classes:', names)

## 5. Choose the base weights — fine-tuning (never from scratch)

In [ ]:
import os
USE_MALARIA_BASE = False   # True -> transfer from your malaria detector (microscopy features)
malaria_pt = '/content/nexus/backend/models/malaria/malaria.pt'
BASE_WEIGHTS = malaria_pt if (USE_MALARIA_BASE and os.path.exists(malaria_pt)) else 'yolov8s.pt'
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector

In [ ]:
EPOCHS = 100
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
import os
PROJECT = '/content/runs/detect'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/nexus_helminths/runs'; os.makedirs(PROJECT, exist_ok=True)
    print('runs -> Drive:', PROJECT)
except Exception as e:
    print('Drive not mounted, runs stay on /content:', e)
model = YOLO(BASE_WEIGHTS)   # pretrained -> fine-tune; head adapts to the helminth-egg classes
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16,
                      project=PROJECT, name='helminths', pretrained=True, patience=25, plots=True,
                      hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=180,
                      fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = model.val()
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

In [ ]:
# ── 6b. RESUME / CONTINUE if a run disconnects (run INSTEAD of step 6) ──
# Robust: rebuilds the model from its own checkpoint (no 4-vs-80 crash), re-establishes
# the dataset if the runtime reset, tries true resume then continue-from-weights.
import os, glob, yaml
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('Drive:', e)
from ultralytics import YOLO

ROOT = '/content/drive/MyDrive/nexus_helminths'
cks = sorted([p for p in glob.glob(ROOT + '/**/last.pt', recursive=True)
              + glob.glob(ROOT + '/**/best.pt', recursive=True) if os.path.isfile(p)], key=os.path.getmtime)
assert cks, 'No checkpoint under nexus_helminths yet — run step 6 (or step 4 then this).'
ckpt = cks[-1]; print('checkpoint:', ckpt)
ck_names = list(YOLO(ckpt).model.names.values()); print('classes (%d):' % len(ck_names), ck_names)

if 'DATA_YAML' not in globals() or not os.path.exists(globals().get('DATA_YAML', '')):
    have = glob.glob('/content/data/**/data.yaml', recursive=True)
    assert have, 'Dataset missing (runtime reset) — re-run step 4 (Roboflow) first, then rerun this.'
    DATA_YAML = have[-1]
# keep dataset class names aligned to the checkpoint
d = yaml.safe_load(open(DATA_YAML)); dn = d['names']
dn = [dn[k] for k in sorted(dn)] if isinstance(dn, dict) else list(dn)
if len(dn) == len(ck_names) and [str(x).lower() for x in dn] != [str(x).lower() for x in ck_names]:
    d['names'] = ck_names; d['nc'] = len(ck_names); yaml.safe_dump(d, open(DATA_YAML, 'w'), sort_keys=False)

model = None
try:
    print('trying true resume...'); model = YOLO(ckpt); model.train(resume=True)
except Exception as e:
    print('resume unavailable -> continue-from-weights:', str(e)[:100]); model = None
if model is None:
    model = YOLO(ckpt)
    model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
                project=ROOT + '/runs', name='helminths_cont', pretrained=True, patience=25, plots=True,
                hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=180, fliplr=0.5, flipud=0.5,
                mosaic=1.0, translate=0.1, scale=0.5)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export `helminths.pt` (app path + Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/nexus_helminths'; os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    print('Drive not mounted (continuing):', e)
best_hits = [c for c in glob.glob('/content/**/weights/best.pt', recursive=True) if os.path.isfile(c)]
last_hits = [c for c in glob.glob('/content/**/weights/last.pt', recursive=True) if os.path.isfile(c)]
backup    = [c for c in glob.glob((DRIVE or '') + '/**/*.pt', recursive=True) if DRIVE and os.path.isfile(c)]
pool = best_hits or last_hits or backup
if not pool:
    raise FileNotFoundError('No trained weights found — run step 6 (training) first.')
best = sorted(pool, key=os.path.getmtime)[-1]
print('using weights:', best, '->', 'best.pt' if best_hits else ('last.pt' if last_hits else 'Drive backup'))
os.makedirs('/content/nexus/backend/models/helminths', exist_ok=True)
dest = '/content/nexus/backend/models/helminths/helminths.pt'; shutil.copy(best, dest); print('saved to app path:', dest)
if DRIVE:
    shutil.copy(best, os.path.join(DRIVE, 'helminths.pt')); print('backed up to Drive')
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
try:
    YOLO(best).export(format='onnx', opset=12)
except Exception as e:
    print('onnx export skipped:', e)
from google.colab import files
files.download(best)

## 8. Put it in the app
1. Move `helminths.pt` → **`backend/models/helminths/helminths.pt`**, commit, push.
2. Auto-loads for `image_type` ∈ {`helminths`, `parasitology`, `stool`, `ova`, `urine_parasite`}.
3. On Render, build with `INSTALL_ML=1` (≥2 GB) to run the detector; else Claude vision reads the field.

Detected organisms → diseases via `backend/ai_services/helminths_organisms.json`
(hookworm → iron-deficiency anaemia; *Opisthorchis* → cholangiocarcinoma risk; ...).